In [2]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.autograd import gradcheck
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms


# use anderson acceleration to find the equilibrium
from torchdeq.solver import anderson

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [ ]:
torch.manual_seed(42) if not torch.cuda.is_available() else torch.cuda.manual_seed_all(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

Using: cpu


# Deep Equilibrium Models (DEQs)

Instead of stacking many layers like a standard ResNet:
```
x₁ = x₀ + ResBlock(x₀)
x₂ = x₁ + ResBlock(x₁)
...
x₅₀ = x₄₉ + ResBlock(x₄₉)
```

DEQs use one block repeatedly until it reaches equilibrium:
```
z* = z* + ResBlock(z*)  # Solve for the fixed point
```

This is equivalent to an infinitely deep network, but only requires storing the final state z*.

## Why This Works

A ResNet with residual connections approximates an ODE:
$$\frac{dx}{dt} = f(x)$$

With Euler integration over L steps:
$$x_{k+1} = x_k + h \cdot f(x_k)$$

DEQ takes this to the limit: instead of simulating L steps, we find where the system settles (the equilibrium).


## Anderson Acceleration

Finding $z^*$ via fixed-point iteration with basic iteration $z_{k+1} = f(z_k)$ is slow.

Instead of just using the current iterate, Anderson acceleration combines the past m iterates:
$$z_{k+1} = \sum_{i=0}^{m} \alpha_i f(z_{k-i})$$

The weights $\alpha_i$ are chosen to minimize the residual.

Reference: Bai et al. "Deep Equilibrium Models" (NeurIPS 2019)

### ResNet

1. Small weight initialization (std=0.01)
    This ensures the mapping is a contraction: $\|f(z) - f(z')\| < \|z - z'\|$

    If weights are too large, the fixed-point iteration won't converge. The Banach theorem requires $\|\frac{\partial f}{\partial z}\| < 1$.

2. GroupNorm instead of BatchNorm
    BatchNorm uses batch statistics which change during the equilibrium solving process. This breaks convergence.

    GroupNorm computes statistics per sample, so it's stable during iteration.

3. Multiple normalization layers
    Prevents activations from exploding during the iterative solving process.

In [4]:
class ResNet(nn.Module):
    def __init__(self, channels: int, hidden_channels: int, k: int=3, num_groups: int=8):
        super().__init__()
        self.conv_in = nn.Conv2d(channels, hidden_channels, k, padding=k//2, bias=False)
        self.conv_out = nn.Conv2d(hidden_channels, channels, k, padding=k//2, bias=False)

        self.gnorm1 = nn.GroupNorm(num_groups=num_groups, num_channels=hidden_channels)
        self.gnorm2 = nn.GroupNorm(num_groups=num_groups, num_channels=channels)
        self.gnorm3 = nn.GroupNorm(num_groups=num_groups, num_channels=channels)

        # normalize convolutional weights between 0 and 0.01 for DEQ stability
        self.conv_in.weight.data.normal_(0, 0.01)
        self.conv_out.weight.data.normal_(0, 0.01)

    def forward(self, z: torch.Tensor, x: torch.Tensor)-> torch.Tensor:
        y = self.gnorm1(F.relu(self.conv_in(z), inplace=True))
        y = self.gnorm2(x + self.conv_out(y))
        y = self.gnorm3(F.relu(z + y, inplace=True))

        return y

In [5]:
X = torch.randn(10, 64, 10, 10)

f = ResNet(64, 128)
z_star, _, _ = anderson.anderson_solver(lambda Z: f(Z, X), torch.zeros_like(X), max_iter=50, tol=1e-4, beta=1.0)

print("Equilibrium found -", z_star.shape)

Equilibrium found - torch.Size([10, 64, 10, 10])


In [6]:
class DEQ(nn.Module):
    def __init__(self, f, solver, **kwargs):
        super().__init__()
        self.f = f
        self.solver = solver
        self.kwargs = kwargs

    def forward(self, x: torch.Tensor) -> torch.Tensor:        
        with torch.no_grad():
            z, _, _ = self.solver(lambda z: self.f(z, x), torch.zeros_like(x), **self.kwargs)
        z: torch.Tensor = self.f(z, x)

        # setting up VJP
        z0 = z.clone().detach().requires_grad_()
        f0 = self.f(z0, x)

        def backward_hook(grad):
            g, _, _ = self.solver(lambda y: torch.autograd.grad(f0, z0, y, retain_graph=True)[0] + grad, grad, **self.kwargs)

            return g

        z.register_hook(backward_hook)
        return z

In [7]:
# check if DEQ gradients are correct
f = ResNet(2, 2, num_groups=2).double()
deq = DEQ(f, anderson.anderson_solver, tol=1e-10, max_iter=500).double()
gradcheck(deq, torch.randn(1, 2, 3, 3).double().requires_grad_(), eps=1e-5, atol=1e-3, check_undefined_grad=False)

True

In [8]:
# Compare DEQ to standard ResNet

# Standard ResNet
class StandardResNet(nn.Module):
    def __init__(self, channels, hidden_channels, num_blocks=10):
        super().__init__()
        self.blocks = nn.ModuleList([
            ResNet(channels, hidden_channels) 
            for _ in range(num_blocks)
        ])
    
    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return x

# Parameter count comparison
standard = StandardResNet(64, 128, num_blocks=10)
deq = ResNet(64, 128)

print(f"Standard ResNet (10 blocks): {sum(p.numel() for p in standard.parameters()):,} params")
print(f"DEQ (1 block, infinite depth): {sum(p.numel() for p in deq.parameters()):,} params")
print(f"Parameter reduction: {sum(p.numel() for p in standard.parameters()) / sum(p.numel() for p in deq.parameters()):.1f}×")

Standard ResNet (10 blocks): 1,479,680 params
DEQ (1 block, infinite depth): 147,968 params
Parameter reduction: 10.0×


In [ ]:
# Training on CIFAR10
cifar_train = datasets.CIFAR10(".", train=True, download=True, transform=transforms.ToTensor())
cifar_test = datasets.CIFAR10(".", train=False, download=True, transform=transforms.ToTensor())

In [38]:
train_loader = DataLoader(cifar_train, batch_size=100, shuffle=True)
test_loader = DataLoader(cifar_test, batch_size=100, shuffle=False)

In [40]:
channels = 48
f = ResNet(channels, 64)
model = nn.Sequential(nn.Conv2d(3, channels, kernel_size=3, bias=True, padding=1), # turn rgb channels into image fetaures
                      nn.BatchNorm2d(channels), # normalize activations
                      DEQ(f, anderson.anderson_solver,  tol=1e-2, max_iter=25),
                      nn.BatchNorm2d(channels),
                      nn.AvgPool2d(8, 8), # reduce dimensionalility for bottleneck compression
                      nn.Flatten(),
                      nn.Linear(channels*4*4, 10) # outpu layer
                    ).to(device)

In [44]:
def train(model: nn.Module, train_loader: DataLoader, max_epoch: int=5, lr: float=0.001) -> tuple[list[float], list[float]]:
    model.train()
    opt = optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, max_epoch* len(train_loader), eta_min=1e-6)

    total_loss = []
    total_error = []

    for _ in tqdm(range(max_epoch), desc="Training", position=0):
        loss_per_epoch = 0.0
        error_per_epoch = 0.0
        for X, y_true in tqdm(train_loader, desc="Batch", position=1, leave=False):
            X = X.to(device)
            y_true = y_true.to(device)

            opt.zero_grad()

            y_pred: torch.Tensor = model(X)
            loss = crit(y_pred, y_true)

            loss.backward()

            if sum(para.grad.isnan().sum() for para in model.parameters() if para.grad is not None) == 0:
                opt.step()
                lr_scheduler.step()

            loss_per_epoch += loss.item() * X.shape[0]

            error_per_epoch += (y_pred.argmax(dim=1) == y_true).sum().item()

        total_error.append(error_per_epoch / len(train_loader))
        total_loss.append(loss_per_epoch / len(train_loader))

    return total_error, total_loss


def test(model: nn.Module, test_loader: DataLoader) -> tuple[float, float]:
    model.eval()
    crit = nn.CrossEntropyLoss()

    total_loss = 0
    total_error = 0

    for X, y_true in tqdm(test_loader, desc="Testing"):
        X = X.to(device)
        y_true = y_true.to(device)

        y_pred: torch.Tensor = model(X)
        loss = crit(y_pred, y_true)

        total_loss += loss.item() * X.shape[0]

        total_error += (y_pred.argmax(dim=1) == y_true).sum().item()

    return total_error/ len(test_loader), total_loss / len(test_loader)

In [46]:
# train for only one epoch due to heavy workload
train_error, train_loss = train(model, train_loader, max_epoch=1)
print(f"Training Error: {train_error[-1]} | Training Loss: {train_loss[-1]}")

Training:   0%|          | 0/1 [00:00<?, ?it/s]

Batch:   0%|          | 0/500 [00:00<?, ?it/s]

Training Error: 52.022 | Training Loss: 133.68525998592378


In [ ]:
test_error, test_loss = test(model, test_loader)
print(f"Test Error: {test_error} | Test Loss: {test_loss}")